# Ответы на экзаменационные вопросы: Numba

Ноутбук закрывает вопросы **6, 18** и практическую задачу билета **№9 (задача 3)**.


In [1]:
import time
import numpy as np
import numba
from numba import njit, prange

rng = np.random.default_rng(1)
print("numba version:", numba.__version__)


numba version: 0.67.0


## Вопросы 6 и 18. Numba: принципы работы, цели, базовые примеры использования

### Цель

Numba — JIT-компилятор (just-in-time) для Python/NumPy кода. Он превращает "медленные" функции с
явными циклами `for` в машинный код, сравнимый по скорости с C/Fortran, **без** переписывания
кода на другом языке — достаточно навесить декоратор.

### Принцип работы

1. При **первом** вызове декорированной функции с конкретными типами аргументов Numba анализирует
   байт-код функции и компилирует специализированную машинную версию именно под эти типы
   (LLVM-компилятор). Это и означает "just-in-time" — компиляция происходит во время выполнения,
   а не заранее.
2. Результат компиляции кэшируется: повторные вызовы с тем же набором типов используют готовый
   машинный код, поэтому первый вызов заметно медленнее последующих ("прогрев").
3. Для типов, поддерживаемых Numba (числа, массивы NumPy, кортежи, некоторые структуры) весь
   Python-цикл выполняется без обращений к интерпретатору CPython и без GIL (если указан
   `nogil=True`), что даёт ускорение, недостижимое для чистого Python.

### Основные способы применения / поддерживаемые режимы

- `@njit` (= `@jit(nopython=True)`) — основной режим: код компилируется полностью, без
  использования объектов Python (только типы, понятные Numba). Если это невозможно — ошибка
  компиляции (без "тихого" отката на медленный Python).
- `@jit` без `nopython=True` — компилятор пытается nopython-режим, а если не получается, тихо
  откатывается на медленный "object mode" (для нового кода обычно не рекомендуется).
- `parallel=True` + `numba.prange` — автоматическая параллелизация независимых итераций цикла по
  ядрам CPU.
- `@vectorize` / `@guvectorize` — превращают скалярную/векторную функцию в настоящий ufunc NumPy
  (в отличие от `np.vectorize`, это реальная компиляция, а не скрытый Python-цикл).
- Кэширование на диск: `@njit(cache=True)` — не перекомпилировать функцию при каждом перезапуске
  процесса.

### Что поддерживается

Числа (`int`, `float`, `complex`), массивы NumPy и операции над ними, циклы, условия, рекурсия,
кортежи, срезы. **Не поддерживаются** (в `nopython`-режиме) произвольные объекты Python общего
вида: списки смешанных типов, словари произвольной структуры, Pandas DataFrame, работа с файлами,
большинство сторонних библиотек.

In [2]:
def sum_squares_py(arr):
    total = 0.0
    for x in arr:
        total += x * x
    return total


@njit
def sum_squares_numba(arr):
    total = 0.0
    for x in arr:
        total += x * x
    return total


data = rng.random(3_000_000)

# "прогрев" — первый вызов включает время компиляции
sum_squares_numba(data[:10])

t0 = time.perf_counter()
r_py = sum_squares_py(data)
t1 = time.perf_counter()
r_nb = sum_squares_numba(data)
t2 = time.perf_counter()
r_np = np.sum(data * data)
t3 = time.perf_counter()

print(f"чистый Python:  {t1 - t0:.4f} c, результат={r_py:.3f}")
print(f"@njit (Numba):  {t2 - t1:.4f} c, результат={r_nb:.3f}")
print(f"NumPy ufunc:    {t3 - t2:.4f} c, результат={r_np:.3f}")
print(f"\nускорение Numba относительно чистого Python: {(t1 - t0) / (t2 - t1):.1f}x")


чистый Python:  0.1510 c, результат=999566.174
@njit (Numba):  0.0018 c, результат=999566.174
NumPy ufunc:    0.0048 c, результат=999566.174

ускорение Numba относительно чистого Python: 84.3x


In [3]:
# parallel=True + prange: распараллеливание независимых итераций
@njit(parallel=True)
def sum_squares_parallel(arr):
    total = 0.0
    for i in prange(arr.size):
        total += arr[i] * arr[i]
    return total


sum_squares_parallel(data[:10])  # прогрев
t0 = time.perf_counter()
r_par = sum_squares_parallel(data)
t1 = time.perf_counter()
print(f"@njit(parallel=True): {t1 - t0:.4f} c, результат={r_par:.3f}")

# guvectorize: превращаем скалярную функцию в настоящий (скомпилированный) ufunc
from numba import guvectorize, float64

@guvectorize([(float64[:], float64[:])], "(n)->()")
def row_norm_gu(row, out):
    s = 0.0
    for v in row:
        s += v * v
    out[0] = s ** 0.5

mat = rng.normal(size=(5, 4))
print("\nнормы строк через @guvectorize:", row_norm_gu(mat).round(3))
print("сравнение с np.linalg.norm:      ", np.linalg.norm(mat, axis=1).round(3))


@njit(parallel=True): 0.0004 c, результат=999566.174

нормы строк через @guvectorize: [2.424 2.48  2.947 0.982 1.638]
сравнение с np.linalg.norm:       [2.424 2.48  2.947 0.982 1.638]


## Практика: билет №9, задача 3

> Приблизительно (с погрешностью порядка 1%) рассчитать, на какой части прямоугольника
> `x ∈ [0, 5], y ∈ [0, 5]` значение функции `z(x, y) = x·y·sin(x)·cos(y)` больше 0.25. Решить
> средствами NumPy/Pandas без циклов Python (а для сравнения — той же задачей через Numba).

### Идея решения

Задача сводится к оценке доли площади, на которой `z(x, y) > 0.25`, методом Монте-Карло: генерируем
много случайных точек в прямоугольнике, считаем долю точек, где условие выполняется — это и есть
оценка искомой доли площади (при достаточном числе точек погрешность оценки убывает как `1/√N`,
для точности ~1% нужно порядка `10⁴–10⁵` точек).

In [4]:
N = 2_000_000
x = rng.uniform(0, 5, size=N)
y = rng.uniform(0, 5, size=N)

z = x * y * np.sin(x) * np.cos(y)          # полностью векторизованно, без циклов Python
fraction_numpy = np.mean(z > 0.25)

print(f"Доля площади прямоугольника, где z(x, y) > 0.25 (NumPy, Монте-Карло, N={N}): {fraction_numpy:.4f}")

# оценка погрешности метода Монте-Карло для доли p: std ~ sqrt(p*(1-p)/N)
p = fraction_numpy
mc_error = (p * (1 - p) / N) ** 0.5
print(f"стандартная ошибка оценки: {mc_error:.5f} (много меньше 1%)")


Доля площади прямоугольника, где z(x, y) > 0.25 (NumPy, Монте-Карло, N=2000000): 0.3514
стандартная ошибка оценки: 0.00034 (много меньше 1%)


In [5]:
@njit(parallel=True)
def monte_carlo_fraction_numba(n, seed):
    np.random.seed(seed)
    count = 0
    for i in prange(n):
        xi = np.random.uniform(0.0, 5.0)
        yi = np.random.uniform(0.0, 5.0)
        zi = xi * yi * np.sin(xi) * np.cos(yi)
        if zi > 0.25:
            count += 1
    return count / n


monte_carlo_fraction_numba(1000, 0)  # прогрев (компиляция)

t0 = time.perf_counter()
fraction_numba = monte_carlo_fraction_numba(N, 123)
t1 = time.perf_counter()
print(f"Доля площади (Numba, явный цикл по N={N} точкам): {fraction_numba:.4f}")
print(f"время выполнения Numba-версии: {t1 - t0:.3f} c")
print("\nОбе оценки согласуются в пределах погрешности метода Монте-Карло:",
      abs(fraction_numpy - fraction_numba) < 0.01)


Доля площади (Numba, явный цикл по N=2000000 точкам): 0.3515
время выполнения Numba-версии: 0.008 c

Обе оценки согласуются в пределах погрешности метода Монте-Карло: True
